# Self-Interacting Neutrinos (SINu) — Massless Neutrino Case

This notebook demonstrates the effect of neutrino self-interactions on the CMB power spectra and matter power spectrum for a cosmology with **massless neutrinos** (N_ur = 3.044, no massive species).

The self-interaction is parametrised by `log10_G_eff_nu`, the base-10 logarithm of the effective
Fermi-like coupling constant $G_\mathrm{eff}$ (in units of $\mathrm{GeV}^{-2}$). The interaction
adds collision terms to the neutrino Boltzmann hierarchy that damp free-streaming and isotropise
the distribution on scales smaller than the neutrino mean free path.

Reference: Camarena, Fogli, Lesgourgues, Smirnov, [arXiv:2309.03941](https://arxiv.org/abs/2309.03941)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from classy import Class

plt.rcParams.update({'font.size': 13, 'figure.dpi': 100})

## Cosmological parameters

We use Planck 2018 best-fit parameters throughout. The SINu coupling is varied from
disabled (vanilla) to three values spanning weak to strong self-interaction.

In [ ]:
# Common cosmological parameters (Planck 2018 best-fit)
common = {
    'output': 'tCl,pCl,lCl,mPk',
    'lensing': 'yes',
    'gauge': 'synchronous',
    'h': 0.6732,
    'omega_b': 0.022383,
    'omega_cdm': 0.12011,
    'A_s': 2.1005e-9,
    'n_s': 0.96605,
    'tau_reio': 0.0543,
    'N_ur': 3.044,      # all neutrinos massless
    'N_ncdm': 0,
    'P_k_max_h/Mpc': 1.0,
    'z_pk': 0,
    'l_max_scalars': 2500,
}

# SINu coupling values to explore
# log10_G_eff_nu: None = SINu off (vanilla), otherwise log10(G_eff [GeV^-2])
cases = [
    {'label': 'SINu off (vanilla)',    'log10_G': None,  'color': 'k',        'ls': '-'},
    {'label': r'$\log_{10}G_\mathrm{eff}=-2.0$', 'log10_G': -2.0, 'color': 'royalblue', 'ls': '--'},
    {'label': r'$\log_{10}G_\mathrm{eff}=-1.5$', 'log10_G': -1.5, 'color': 'tomato',    'ls': '-.'},
    {'label': r'$\log_{10}G_\mathrm{eff}=-1.0$', 'log10_G': -1.0, 'color': 'seagreen',  'ls': ':'},
]

## Run CLASS for each case

In [ ]:
results = []

for case in cases:
    params = dict(common)
    if case['log10_G'] is not None:
        params['log10_G_eff_nu'] = case['log10_G']
        params['interacting_nu'] = 1

    print(f"Computing: {case['label']} ...", end=' ', flush=True)
    cosmo = Class()
    cosmo.set(params)
    cosmo.compute()

    h = cosmo.h()
    kvec = np.logspace(-3, 0, 500)          # k in h/Mpc
    pk   = cosmo.get_pk_all(kvec * h, 0.) * h**3  # P(k) in (Mpc/h)^3

    cls  = cosmo.lensed_cl(2500)
    ell  = np.array(cls['ell'][2:], dtype=float)
    cl_tt = np.array(cls['tt'][2:]) * ell * (ell + 1) / (2 * np.pi)
    cl_ee = np.array(cls['ee'][2:]) * ell * (ell + 1) / (2 * np.pi)

    results.append({'kvec': kvec, 'pk': pk, 'ell': ell,
                    'cl_tt': cl_tt, 'cl_ee': cl_ee, **case})

    cosmo.struct_cleanup()
    cosmo.empty()
    print('done')

## CMB Temperature Power Spectrum $C_\ell^{TT}$

The left panel shows the absolute $D_\ell^{TT} = \ell(\ell+1)C_\ell^{TT}/(2\pi)$ spectra.
The right panel shows the ratio relative to the vanilla (SINu-off) case, revealing the
small but non-zero effect of neutrino self-interactions on the CMB anisotropies.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

vanilla = results[0]

for r in results:
    axes[0].plot(r['ell'], r['cl_tt'], color=r['color'], ls=r['ls'], label=r['label'])
    axes[1].plot(r['ell'], r['cl_tt'] / vanilla['cl_tt'],
                 color=r['color'], ls=r['ls'], label=r['label'])

axes[0].set_xscale('log')
axes[0].set_xlabel(r'Multipole $\ell$')
axes[0].set_ylabel(r'$D_\ell^{TT}\;[\mu\mathrm{K}^2]$  (dimensionless units)')
axes[0].set_title('CMB TT power spectrum')
axes[0].legend(fontsize=11)
axes[0].set_xlim(2, 2500)

axes[1].axhline(1, color='k', lw=0.8, ls='--')
axes[1].set_xscale('log')
axes[1].set_xlabel(r'Multipole $\ell$')
axes[1].set_ylabel(r'$D_\ell^{TT}\,(\mathrm{SINu}) / D_\ell^{TT}\,(\mathrm{vanilla})$')
axes[1].set_title('Ratio to vanilla')
axes[1].legend(fontsize=11)
axes[1].set_xlim(2, 2500)

fig.tight_layout()
plt.show()

## CMB E-mode Polarisation Power Spectrum $C_\ell^{EE}$

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for r in results:
    axes[0].plot(r['ell'], r['cl_ee'], color=r['color'], ls=r['ls'], label=r['label'])
    axes[1].plot(r['ell'], r['cl_ee'] / vanilla['cl_ee'],
                 color=r['color'], ls=r['ls'], label=r['label'])

axes[0].set_xscale('log')
axes[0].set_xlabel(r'Multipole $\ell$')
axes[0].set_ylabel(r'$D_\ell^{EE}$ (dimensionless units)')
axes[0].set_title('CMB EE power spectrum')
axes[0].legend(fontsize=11)
axes[0].set_xlim(2, 2500)

axes[1].axhline(1, color='k', lw=0.8, ls='--')
axes[1].set_xscale('log')
axes[1].set_xlabel(r'Multipole $\ell$')
axes[1].set_ylabel(r'$D_\ell^{EE}\,(\mathrm{SINu}) / D_\ell^{EE}\,(\mathrm{vanilla})$')
axes[1].set_title('Ratio to vanilla')
axes[1].legend(fontsize=11)
axes[1].set_xlim(2, 2500)

fig.tight_layout()
plt.show()

## Matter Power Spectrum $P(k)$

Neutrino self-interactions suppress free-streaming, which slightly increases the neutrino
clustering on small scales, enhancing $P(k)$ at large $k$.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for r in results:
    axes[0].loglog(r['kvec'], r['pk'], color=r['color'], ls=r['ls'], label=r['label'])
    axes[1].semilogx(r['kvec'], r['pk'] / vanilla['pk'],
                     color=r['color'], ls=r['ls'], label=r['label'])

axes[0].set_xlabel(r'$k\;[h/\mathrm{Mpc}]$')
axes[0].set_ylabel(r'$P(k)\;[(\mathrm{Mpc}/h)^3]$')
axes[0].set_title('Matter power spectrum')
axes[0].legend(fontsize=11)

axes[1].axhline(1, color='k', lw=0.8, ls='--')
axes[1].set_xlabel(r'$k\;[h/\mathrm{Mpc}]$')
axes[1].set_ylabel(r'$P(k)\,(\mathrm{SINu})\,/\,P(k)\,(\mathrm{vanilla})$')
axes[1].set_title('Ratio to vanilla')
axes[1].legend(fontsize=11)

fig.tight_layout()
plt.show()

## Summary

- **CMB**: SINu affects the acoustic peak heights and phase at the sub-percent level. The effect
  grows with coupling strength and is largest at intermediate to high multipoles.
- **P(k)**: Self-interactions suppress free-streaming, reducing neutrino anisotropic stress.
  On small scales this shifts power relative to the vanilla case.
- To enable SINu in your own code, set `log10_G_eff_nu` (and optionally `interacting_nu = 1`).
  The default (`log10_G_eff_nu < -6`) corresponds to no self-interaction.